In [ ]:
# 1. Carregar bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# --- 1. Carregar o dataset
url = "https://github.com/maikisoares00/Projeto_Aplicado_III_RetailWise/blob/main/dataset/dataset_varejo.xlsx"
url_raw = url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

df = pd.read_excel(url_raw, engine="openpyxl")

In [ ]:
# --- 2. Ajuste de tipos
df["UnitPrice"] = (
    df["UnitPrice"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce").fillna(0).astype(int)

In [ ]:
# --- 3. Remover dados ausentes em CustomerID
df = df.dropna(subset=["CustomerID"])


In [ ]:
#4. Eliminar transações inválidas
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

In [ ]:
# --- 5. Filtrar apenas as colunas relevantes
df_model = df[["CustomerID", "StockCode", "Quantity"]]

In [ ]:
# --- 6. Criar matriz usuário-produto
user_item_matrix = pd.pivot_table(
    df_model,
    index="CustomerID",
    columns="StockCode",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

In [ ]:
 #--- 7. (Opcional) Filtrar subconjunto — clientes com mais de 5 transações
clientes_ativos = df_model["CustomerID"].value_counts()
clientes_filtrados = clientes_ativos[clientes_ativos > 5].index
user_item_matrix = user_item_matrix.loc[user_item_matrix.index.isin(clientes_filtrados)]

In [ ]:
# --- 8. (Opcional) Normalizar a matriz (0 a 1)
from sklearn.preprocessing import MinMaxScaler  # <-- Import obrigatório

scaler = MinMaxScaler()
user_item_matrix.columns = user_item_matrix.columns.astype(str) # Convert column names to strings
user_item_matrix_norm = pd.DataFrame(
    scaler.fit_transform(user_item_matrix),
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)

print("Matriz normalizada com sucesso!")
display(user_item_matrix_norm.head())

Matriz normalizada com sucesso!


StockCode,10002,10080,10120,10125,10133,10135,11001,15030,15034,15036,...,90214V,90214W,90214Y,90214Z,BANK CHARGES,C2,DOT,M,PADS,POST
CustomerID,,,,,,,,,,,,,,,,,,,,,
12347.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
12348.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.084112
12349.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.009346
12350.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.009346
12352.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000677,0.0,0.065421


In [ ]:
# --- 9. Exibir resultado final
print("✅ Base tratada e matriz usuário-produto criada!")
print(f"Linhas (usuários): {user_item_matrix_norm.shape[0]}")
print(f"Colunas (produtos): {user_item_matrix_norm.shape[1]}")
display(user_item_matrix_norm.head())

✅ Base tratada e matriz usuário-produto criada!
Linhas (usuários): 4039
Colunas (produtos): 3665


StockCode,10002,10080,10120,10125,10133,10135,11001,15030,15034,15036,...,90214V,90214W,90214Y,90214Z,BANK CHARGES,C2,DOT,M,PADS,POST
CustomerID,,,,,,,,,,,,,,,,,,,,,
12347.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
12348.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.084112
12349.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.009346
12350.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.009346
12352.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000677,0.0,0.065421


In [ ]:
df_model.to_csv('df_model.csv', index=True)